In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read from Bronze

In [0]:
log("Reading from Bronze ...")
df_bronze = spark.table(TBL_BRONZE_RAW)

In [0]:
display(df_bronze.limit(5))

## 2. Extract Product Columns & Add Ingestion Metadata

In [0]:
df_products = df_bronze.select(
    F.col("product_id"),
    F.col("product_name"),
    F.col("category"),
    F.col("sub_category"),
    F.col("ingested_at"),
    F.col("file_path"),
    F.col("file_name"),
    F.col("file_size")
    ) \
    .withColumn("transformed_at", F.current_timestamp()) 
display(df_products)

## 3. Deduplicate

In [0]:
df_duplicate = df_products.groupBy("product_id").count().filter(F.col("count")>1)
df_duplicate.show()

In [0]:
log(f"Row before duplicate drop: {df_products.count():,}")
df_silver_products = df_products.dropDuplicates(["product_id"])
log(f"Row after duplicate drop: {df_silver_products.count():,}")

In [0]:
display(df_silver_products)

## 4. Standardize Category Values

In [0]:
df_silver_products = df_silver_products \
  .withColumn("category", F.initcap(F.col("category"))) \
  .withColumn("sub_category", F.initcap(F.col("sub_category")))

In [0]:
display(df_silver_products.limit(10))

## 5. Write to Silver

In [0]:
df_silver_products.write.format("delta").option("mergeSchema", "true").option("delta.enableChangeDataFeed", "true").mode("overwrite").saveAsTable(TBL_SILVER_PRODUCTS)

In [0]:
display(spark.table(TBL_SILVER_PRODUCTS).limit(10))